#### Function calling
- OpenAI 모델이 사용자 코드 또는 외부 서비스와 상호 작용할 수 있도록 지원하는 기능
- LLM 모델은 데이터베이스 조회, 서버 API 호출, 파일 저장 등의 작업을 수행하지 못함
    => 실제 필요한 작업에 대해 함수 형태로 작성 후 호출이 필요

In [7]:
from dotenv import load_dotenv, find_dotenv
# .env 파일 정보 가져오기
load_dotenv(find_dotenv())

True

In [8]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5-nano",
    input="2026년 2월 20일 삼성전자 주가를 알려줘"
)

print(response.output_text)

죄송하지만, 저는 실시간 데이터나 특정 과거 날짜의 주가를 바로 조회해 드릴 수 있는 기능이 없어요. 다만 2026-02-20일의 삼성전자 주가를 직접 확인하는 방법을 안내해 드리겠습니다.

주가 확인 방법
- Yahoo Finance
  - 티커: 005930.KS
  - Historical Data(과거 데이터)에서 날짜 범위를 2026-02-20으로 설정
  - 종가(Close) 또는 수정종가(Adj Close) 확인
- Investing.com 또는 Google Finance
  - 삼성전자(005930.KS) 검색 → Historical Data/과거 데이터 메뉴에서 날짜 선택
- 국내 증권사 앱/홈페이지
  - 해외주식이 아닌 국내 거래소(KRX) 기준으로 날짜 선택 후 종가 확인

원하시면 제가 직접 사용할 수 있는 간단한 코드도 드릴게요(예: Python + yfinance로 2026-02-20의 종가 가져오기). 필요하시면 말씀해 주세요. 또한 확인한 주가를 알려주시면, 간단한 분석(전일 대비 변화율, 등락폭 등)도 도와드리겠습니다.


In [ ]:
# 야후 파이낸스 호출 => 종목 코드로만 조회 가능

# !pip install yfinance

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 41.0 MB/s  0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15690 sha256=a58e3c053c17209df0855193d8b3f9a75ab1b1f7bcad608d81496de83a206e4c
  Stored in directory: c:\users\soldesk\appdata\local\pip\cache\wheels\7e\62\f9\20d7dbb144b6f563edab8e3a7fda71d976870cd41972035cdd
Successfully built multitasking

   ---------------------------------------- 0/7 [peewee]
   ---------------------------------------- 0/7 [peewee]
   ----------- ---------------------------- 2/7 [websockets]
   ----------- ----------


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import yfinance as yf

# 삼성전자
stock = yf.Ticker("005930.KS")
data = stock.history(start="2026-02-20")
print(data)

                               Open      High       Low     Close    Volume  \
Date                                                                          
2026-02-20 00:00:00+09:00  190000.0  190300.0  188600.0  190100.0  24213880   
2026-02-23 00:00:00+09:00  194400.0  197600.0  194300.0  195300.0   9075932   

                           Dividends  Stock Splits  
Date                                                
2026-02-20 00:00:00+09:00        0.0           0.0  
2026-02-23 00:00:00+09:00        0.0           0.0  


In [11]:
# 1. 스키마 정의 (JSON 스키마)
# 2. 함수 작성 
import json


tools = [
    {
        "type":"function",
        "name":"get_korea_stock_price",
        "description":"한국 주식 종목코드를 받아 현재 가격을 조회합니다.",
        "parameters":{
            "type":"object",
            "properties":{
                "code":{
                    "type":"string",
                    "description":"6자리 한국 주식 코드 (예: 005930)"
                }
            },
            "required":["code"]
        }
    }
]
def get_korea_stock_price(code):
    symbol = f"{code}.KS"

    stock = yf.Ticker(symbol)
    data = stock.history(period="1d")

    if data.empty:
        return json.dumps({"error":"종목코드를 확인해 주세요"})
    
    latest = data.iloc[-1]

    return json.dumps({
        "code":code,
        "current_price":int(latest['Close']),
        "open":int(latest['Open']),
        "high":int(latest['High']),
        "low":int(latest['Low']),
        "volume":int(latest['Volume'])
    })

def run_korea_stock_agent(prompt):
    # prompt 값에 따라 tools 사용 여부를 모델이 판단
    response = client.responses.create(
        model="gpt-5-nano",
        tools=tools,
        input=[
            {"role":"user", "content":prompt}
        ]
    )
    # function 호출 결과만 수집
    function_calls = [item for item in response.output if item.type == "function_call"]

    if not function_calls:
        return response.output_text
    
    # function_calls 이 여러개 일 수도 있기 때문에 담아둔다 
    tool_outputs = []

    for call in function_calls:
        args = json.loads(call.arguments)
        result = get_korea_stock_price(**args)

        tool_outputs.append({
            "type":"function_call_output",
            "call_id":call.call_id, # 어떤 function_call 에 대한 결과인지에 대한 매핑 처리
            "output":result
        })
    final_response = client.responses.create(
        model="gpt-5-nano",
        tools=tools,
        input=[
            {"role":"user", "content":prompt},
            *response.output, *tool_outputs
        ]
    )

    return final_response
    

In [12]:
run_korea_stock_agent("삼성전자의 현재 주가를 알려줘")

Response(id='resp_077a4be769148b6100699bc1b32f80819680fba23aacbee819', created_at=1771815347.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_077a4be769148b6100699bc1b3a9ac81968594554cb94affe5', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_077a4be769148b6100699bc1b5fe548196ae28b541f0f47b86', content=[ResponseOutputText(annotations=[], text='삼성전자(005930) 현재가: 194,000원\n\n오늘 정보 요약:\n- 시가: 194,400원\n- 고가: 197,600원\n- 저가: 192,250원\n- 거래량: 12,278,980주\n\n참고: 주가 정보는 지연될 수 있으며, 실시간 가격은 거래소 상황에 따라 변동합니다. 더 자세한 정보나 알림 설정을 원하시면 알려 주세요.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_korea_stock_price', parameters={'type': 'object', 'properties': {'code': {'type': 'string', 'description'

In [13]:
print(run_korea_stock_agent("005930의 현재 주가를 알려줘"))

Response(id='resp_0c889cadfd2c42fb00699bc1ef4d488196940fbcbc95f9b142', created_at=1771815407.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0c889cadfd2c42fb00699bc1effec4819695b40ea8f5963ec4', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0c889cadfd2c42fb00699bc1f308d081969103d0707ebd4904', content=[ResponseOutputText(annotations=[], text='005930(삼성전자)의 현재가: 194,200원입니다.\n- 시가: 194,400원\n- 고가: 197,600원\n- 저가: 192,250원\n- 거래량: 12,331,812주\n\n다른 정보도 원하시면 말씀해 주세요.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_korea_stock_price', parameters={'type': 'object', 'properties': {'code': {'type': 'string', 'description': '6자리 한국 주식 코드 (예: 005930)'}}, 'required': ['code'], 'additional

In [28]:
from openai import OpenAI
import json
import requests

client = OpenAI()

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시/loaction 을 받아 현재 날씨(기온, 풍속 등)를 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "예: 'Seoul', 또는 '서울' 같은 도시/지역명 입력",
                },
                "timezone":{
                    "type":"string",
                    "description":"예: 'Asia/Seoul' 또는 생략 가능",
                    "default":"auto"
                }
            },
            "required": ["location"],
        },
    },
]

def get_weather(location, timezone):
    # 위도, 경도 찾기
    params = {"name":location, "count":1, "language":"ko"}
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search", params=params, timeout=30).json()
    results = geo.get("results")[0]

    if not results:
        return json.dumps({"error": f"{location} 위치를 찾기 못했습니다"})
    lat, lon = results.get("latitude"), results.get("longitude")

    print(f"위도 {lat}, 경도 {lon}")
    # 날씨 api 호출
    url = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude":lat,
        "longitude":lon,
        "current_weather":"true",
        "timezone":timezone
    }
    weather_result = requests.get(url, params=weather_params, timeout=30).json()
    current_weather = weather_result.get("current_weather")

    if not current_weather:
        return json.dumps({"error" : "날씨 데이터를 가져오지 못했습니다"})
    # 받은 결과를 모델에게 돌려주기 위한 json 구조 설정
    return json.dumps({
        "loaction":location,
        "latitude":lat,
        "longitude":lon,
        "temperature_c":current_weather.get("temperature"),
        "wind_speed_kmh":current_weather.get("wind_speed"),
        "winddirection_deg":current_weather.get("winddirection"),
        "weathercode":current_weather.get("weathercode"),
        "time":current_weather.get("time")
    })
def execute_tool(name, arguments):
    if name == "get_weather":
        return get_weather(**arguments)
    
def run_weather(prompt):

    input_list = [
        {"role": "user", "content": prompt}
    ]

    response = client.responses.create(
    model="gpt-5-nano",
    tools=tools,
    input=input_list,
    )
# Create a running input list we will add to over time

# function 호출 결과만 수집
    function_calls = [item for item in response.output if item.type == "function_call"]

    if not function_calls:
        return response.output_text
    
    # function_calls 이 여러개 일 수도 있기 때문에 담아둔다 
    tool_outputs = []

    for call in function_calls:
        args = json.loads(call.arguments)
        result = execute_tool(call.name, args)

        tool_outputs.append({
            "type":"function_call_output",
            "call_id":call.call_id, # 어떤 function_call 에 대한 결과인지에 대한 매핑 처리
            "output":result
        })
    final_response = client.responses.create(
        model="gpt-5-nano",
        tools=tools,
        input=[
             {"role": "user", "content": prompt},
            *response.output, *tool_outputs
        ]
    )

    return final_response



In [29]:
run_weather("Seoul 날씨 어때?")

위도 37.566, 경도 126.9784


Response(id='resp_09e0804873f6ddf900699bd506d3248193a50bb590117b175e', created_at=1771820294.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_09e0804873f6ddf900699bd507af2081939c7819ebf651dfb9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_09e0804873f6ddf900699bd50e14148193aab9acf916827067', content=[ResponseOutputText(annotations=[], text='지금 서울 날씨 요약:\n\n- 기온: 약 1.5°C\n- 바람: 북서(NW) 방향으로 부는 중\n- 시각: 2026-02-23 13:15\n\n풍속 정보는 확인되지 않아 바람 강도는 알 수 없습니다. 강수 여부나 구름 상태 등 자세한 정보가 필요하면 더 자세한 예보를 조회해 드릴게요. 24시간 예보를 원하시나요?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'location': {'type': 'string', 'description': "예: 'Seoul', 또는 